# Edge–Cloud Collaborative Scheduling Lab

This notebook is a runnable companion to the
[Codeforces 2251A problem](https://codeforces.com/contest/2251/problem/A).
It keeps four activities in one place:

1. understand the system and interactive protocol;
2. connect those concepts to both the frozen baseline and the current layered scheduler;
3. run the policy against deterministic scenarios; and
4. add optimizations one at a time and measure what actually improves.

The repository remains the source of truth. The notebook reads the checked-in C++ source,
scenarios, task-time table, local judge, and baseline benchmark instead of copying them into
a disconnected toy implementation.

## Goal

By the end of this lab, we should be able to answer:

- What work runs on the edge, in a cloud, and on the shared links?
- What does one request do from `ARR` to `FIN`?
- What does an assignment such as `E D PRE -1 3 7 12 19` mean?
- Why is the frozen baseline correct but intentionally inefficient?
- When does decode grouping help, and when can waiting for a group hurt?
- Which scenario should expose each optimization?
- Did a code change remain legal, and did it improve score, throughput, TDR, or TPOT?

**Notebook mode:** tutorial + experiment log.  
**Reader:** someone learning the problem while implementing a contest scheduler.  
**Handoff:** a top-to-bottom executable notebook tied to the current repository.

## Setup

In [1]:
from __future__ import annotations

import html
import json
import math
import re
import subprocess
from pathlib import Path
from typing import Any, Iterable

from IPython.display import Code, HTML, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts in the root or notebooks/ directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "main.cpp").is_file() and (candidate / "tools/local_judge.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find main.cpp and tools/local_judge.py above the working directory")


REPO_ROOT = find_repo_root()
BUILD_DIR = REPO_ROOT / "build"
BASELINE_SOLVER = BUILD_DIR / "v0-baseline"
WORKING_SOLVER = BUILD_DIR / "scheduler"
SCENARIO_DIR = REPO_ROOT / "scenarios"
BASELINE_SNAPSHOT = REPO_ROOT / "benchmarks/baseline-v0.json"
REGISTRY_PATH = REPO_ROOT / "scheduler_versions/registry.json"

print(f"Repository: {REPO_ROOT}")
print(f"Frozen v0:  {BASELINE_SOLVER}")
print(f"Current v7: {WORKING_SOLVER}")

Repository: /Users/aadikhanna/Github/AI Optimizer/CodeForces-Edge-Collaborative-Scheduling
Frozen v0:  /Users/aadikhanna/Github/AI Optimizer/CodeForces-Edge-Collaborative-Scheduling/build/v0-baseline
Current v7: /Users/aadikhanna/Github/AI Optimizer/CodeForces-Edge-Collaborative-Scheduling/build/scheduler


In [2]:
def run_checked(command: list[str], timeout_seconds: float = 120.0) -> subprocess.CompletedProcess[str]:
    """Run a bounded command in the repository and show concise output."""
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        timeout=timeout_seconds,
    )
    if completed.stdout.strip():
        print(completed.stdout.rstrip())
    if completed.returncode != 0:
        if completed.stderr.strip():
            print(completed.stderr.rstrip())
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {' '.join(command)}")
    return completed


run_checked(["make", "build/v0-baseline", "build/scheduler"])
assert BASELINE_SOLVER.is_file(), "The frozen baseline executable was not created"
assert WORKING_SOLVER.is_file(), "The current scheduler executable was not created"

make[1]: `build/v0-baseline' is up to date.
make[1]: `build/scheduler' is up to date.


In [3]:
def display_table(rows: Iterable[dict[str, Any]], columns: list[tuple[str, str]] | None = None) -> None:
    """Render a small list of dictionaries without requiring pandas."""
    bounded_rows = list(rows)
    if not bounded_rows:
        display(Markdown("_No rows._"))
        return
    if columns is None:
        columns = [(key, key) for key in bounded_rows[0]]
    header = "".join(f"<th>{html.escape(label)}</th>" for _, label in columns)
    body = []
    for row in bounded_rows:
        cells = "".join(
            f"<td>{html.escape(str(row.get(key, '')))}</td>" for key, _ in columns
        )
        body.append(f"<tr>{cells}</tr>")
    display(
        HTML(
            "<table><thead><tr>"
            + header
            + "</tr></thead><tbody>"
            + "".join(body)
            + "</tbody></table>"
        )
    )

## 1. Build the mental model

Each request has a **prefill phase** followed by one or more **decode iterations**.

```text
ARR
  │
  ▼
Edge:  P PRE ──UP──▶ Cloud: P PROC ──DOWN──▶ Edge: P POST
                                                   │
                                                   ▼
Edge:  D PRE ──UP──▶ Cloud: D PROC ──DOWN──▶ Edge: D POST
         ▲                                           │
         └──────── next token if not FIN ────────────┘
```

Resource constraints:

- There is one edge compute server, `E`.
- There are `K` cloud compute servers, `C0 ... C(K-1)`.
- Every server can run at most one task at a time.
- All clouds collectively share one FIFO `UP` transfer queue and one FIFO `DOWN` queue.
- Transfers do not occupy edge or cloud compute, but competing transfers queue on their link.
- A request is assigned to a cloud by its `P PRE` task and keeps that cloud association.

### Prefill versus decode grouping

| Family | Edge stage | Cloud stage | Groupable? |
|---|---|---|---|
| Prefill (`P`) | `P PRE`, `P POST` | `P PROC` | No: each assignment names one request |
| Decode (`D`) | `D PRE`, `D POST` | `D PROC` | Yes: assignments carry a list of request IDs |

Grouping is therefore **not cloud-only**. Decode preprocessing and postprocessing can be
grouped on the edge, while decode processing can be grouped on a cloud. A `D PROC` group
must contain requests assigned to that same cloud.

A group is also **not permanent**. It represents one stage of one decode iteration. After
`D POST`, unfinished requests become eligible for their next token and may be regrouped.
A request that reaches its hidden output length emits `FIN` and is absent from future groups.

### Reading one assignment

```text
E D PRE -1 3 7 12 19
│ │  │   │ │ └────── request IDs in this group
│ │  │   │ └──────── group size = 3
│ │  │   └────────── required placeholder for edge decode work
│ │  └────────────── preprocessing stage
│ └───────────────── decode family
└─────────────────── run on the edge server
```

This starts one grouped decode-preprocessing task for requests `7`, `12`, and `19`.
It does not mean “run from time 7 to time 19,” and `-1` is not a cloud ID.

## 2. Inspect the workload suite

Scenario JSON includes system parameters, scoring weights, a task-time table, and requests.
`output_length` is judge-only truth: the local judge uses it to decide when to emit `FIN`,
but the scheduler only sees `ARR request_id input_length`.

In [4]:
scenario_paths = sorted(SCENARIO_DIR.glob("*.json"))
scenarios = {path.stem: json.loads(path.read_text()) for path in scenario_paths}

pressure_by_name = {
    "official_worked_example": "calibration",
    "single_sanity": "one-request lifecycle",
    "two_cloud_parallel": "parallelism / reservations",
    "output_length_skew": "hidden output skew",
    "batch_friendly_burst": "decode grouping",
    "latency_sensitive_stream": "throughput vs latency",
    "link_bottleneck": "shared UP/DOWN queues",
    "prefill_preemption": "prefill layer chunking",
    "degenerate_one_layer": "minimum legal mechanics",
    "interpolation_missing_values": "task-time interpolation",
    "single_cloud_prefill_interleave": "adaptive prefill chunking",
    "slo_priority_collision": "SLO-aware urgency",
    "latency_weighted_slow_link": "latency-weighted link policy",
    "nonmonotonic_batch_table": "table-aware group size",
}

scenario_summary = []
for path in scenario_paths:
    data = scenarios[path.stem]
    requests = data["requests"]
    scenario_summary.append(
        {
            "scenario": data["name"],
            "K": data["system"]["K"],
            "S": data["system"]["S"],
            "requests": len(requests),
            "tokens": sum(request["output_length"] for request in requests),
            "w_tp": data["scoring"]["w_tp"],
            "w_c": data["scoring"]["w_c"],
            "pressure": pressure_by_name.get(data["name"], "general policy behavior"),
        }
    )

display_table(
    scenario_summary,
    [
        ("scenario", "Scenario"),
        ("K", "Clouds"),
        ("S", "Schedule cost"),
        ("requests", "Requests"),
        ("tokens", "Hidden output tokens"),
        ("w_tp", "Throughput weight"),
        ("w_c", "Latency weight"),
        ("pressure", "Designed to expose"),
    ],
)

Scenario,Clouds,Schedule cost,Requests,Hidden output tokens,Throughput weight,Latency weight,Designed to expose
official_worked_example,1,1.0,1,1,0.5,0.5,calibration
single_sanity,1,1.0,1,3,0.5,0.5,one-request lifecycle
two_cloud_parallel,2,1.0,6,29,0.5,0.5,parallelism / reservations
output_length_skew,2,1.5,6,56,0.6,0.4,hidden output skew
batch_friendly_burst,4,8.0,16,128,0.9,0.1,decode grouping
latency_sensitive_stream,3,2.0,12,60,0.2,0.8,throughput vs latency
link_bottleneck,4,1.0,8,32,0.6,0.4,shared UP/DOWN queues
prefill_preemption,2,1.0,5,64,0.5,0.5,prefill layer chunking
degenerate_one_layer,1,1.0,3,6,0.5,0.5,minimum legal mechanics
interpolation_missing_values,2,1.0,4,10,0.5,0.5,task-time interpolation


### Choose a scenario to study

Change `SELECTED_SCENARIO_FILE`, rerun this cell, and then rerun the experiment cells below.

In [5]:
SELECTED_SCENARIO_FILE = "04_batch_friendly_burst.json"

selected_path = SCENARIO_DIR / SELECTED_SCENARIO_FILE
selected = json.loads(selected_path.read_text())

display(Markdown(f"### `{selected['name']}`\n\n{selected['description']}"))
display(Code(json.dumps({"system": selected["system"], "scoring": selected["scoring"]}, indent=2), language="json"))

request_preview = []
for request_id, request in enumerate(selected["requests"][:12]):
    request_preview.append(
        {
            "request_id": request_id,
            "arrival": request["arrival"],
            "input_length (visible)": request["input_length"],
            "output_length (hidden)": request["output_length"],
        }
    )
display_table(request_preview)
if len(selected["requests"]) > len(request_preview):
    print(f"Showing {len(request_preview)} of {len(selected['requests'])} requests.")

### `batch_friendly_burst`

A high-overhead request burst where decode grouping should materially improve throughput.

{
  "system": {
    "K": 4,
    "S": 8.0,
    "latency_in_ms": 0.2,
    "bandwidth_gbps": 100.0,
    "bytes_per_token": 512,
    "num_layers": 8
  },
  "scoring": {
    "SLO1": 600.0,
    "SLO2": 100.0,
    "tp_UB": 0.35,
    "tp_base": 0.01,
    "dist_base": 5.0,
    "w_tp": 0.9,
    "w_c": 0.1
  }
}

request_id,arrival,input_length (visible),output_length (hidden)
0,0.0,16,8
1,0.0,16,8
2,0.0,16,8
3,0.0,16,8
4,0.0,16,8
5,0.0,16,8
6,0.0,16,8
7,0.0,16,8
8,0.0,16,8
9,0.0,16,8


Showing 12 of 16 requests.


## 3. Connect the model to the code

We keep two distinct artifacts:

- `scheduler_versions/v0_baseline.cpp` is the frozen, deliberately simple reference;
- `main.cpp` is the current layer-7 submission and is identical to
  `scheduler_versions/layered_scheduler.cpp` with its default `OPT_LEVEL=7`.

The frozen baseline is a state machine plus three FIFO structures:

- `pending_requests_`: arrived requests that do not yet have a cloud reservation;
- `edge_ready_`: legal edge tasks ordered by when they became ready; and
- `cloud_ready_[cloud]`: legal cloud tasks for each cloud.

It adds one deliberate restriction: it reserves an entire cloud for a request
from `P PRE` until `FIN`. The contest does not require that restriction. It makes the first
implementation easy to reason about, but it leaves clouds idle while their reserved request
is on the edge or waiting for a transfer.

In [6]:
baseline_source = (REPO_ROOT / "scheduler_versions/v0_baseline.cpp").read_text()
layered_source = (REPO_ROOT / "scheduler_versions/layered_scheduler.cpp").read_text()


def source_between(
    source: str, start_marker: str, end_marker: str, max_lines: int = 180
) -> str:
    start = source.index(start_marker)
    end = source.index(end_marker, start)
    snippet = source[start:end].rstrip()
    lines = snippet.splitlines()
    if len(lines) > max_lines:
        lines = lines[:max_lines] + ["// ... bounded notebook preview ..."]
    return "\n".join(lines)


display(Markdown("### Request states"))
display(
    Code(
        source_between(baseline_source, "enum class RequestState", "enum class TaskKind"),
        language="cpp",
    )
)

### Request states

enum class RequestState {
    UNSEEN,
    WAITING_FOR_CLOUD,
    P_PRE_RUNNING,
    WAITING_PREFILL_UP,
    P_PROC_READY,
    P_PROC_RUNNING,
    WAITING_PREFILL_DOWN,
    P_POST_READY,
    P_POST_RUNNING,
    D_PRE_READY,
    D_PRE_RUNNING,
    WAITING_DECODE_UP,
    D_PROC_READY,
    D_PROC_RUNNING,
    WAITING_DECODE_DOWN,
    D_POST_READY,
    D_POST_RUNNING,
    FINISHED,
};

In [7]:
display(Markdown("### Baseline admission and edge dispatch"))
display(
    Code(
        source_between(
            baseline_source, "string dispatch_admission()", "string dispatch_cloud_task"
        ),
        language="cpp",
    )
)

### Baseline admission and edge dispatch

string dispatch_admission() {
        int request_id = pending_requests_.front();
        pending_requests_.pop_front();

        int cloud = free_clouds_.front();
        free_clouds_.pop_front();

        Request& req = request(request_id);
        expect_state(req, RequestState::WAITING_FOR_CLOUD, "P PRE dispatch");
        if (cloud_reserved_[cloud]) {
            fail("admission selected a reserved cloud");
        }

        req.cloud = cloud;
        req.state = RequestState::P_PRE_RUNNING;
        cloud_reserved_[cloud] = true;
        edge_busy_ = true;

        return "E P PRE " + to_string(cloud) + " " + to_string(request_id);
    }

    string dispatch_edge_task() {
        ReadyTask task = edge_ready_.front();
        edge_ready_.pop_front();
        Request& req = request(task.request_id);

        edge_busy_ = true;
        switch (task.kind) {
            case TaskKind::P_POST:
                expect_state(req, RequestState::P_POST_READY, "P POST dispatch");
                req.state = RequestState::P_POST_RUNNING;
                return "E P POST " + to_string(req.cloud) + " " + to_string(req.id);
            case TaskKind::D_PRE:
                expect_state(req, RequestState::D_PRE_READY, "D PRE dispatch");
                req.state = RequestState::D_PRE_RUNNING;
                return "E D PRE -1 1 " + to_string(req.id);
            case TaskKind::D_POST:
                expect_state(req, RequestState::D_POST_READY, "D POST dispatch");
                req.state = RequestState::D_POST_RUNNING;
                return "E D POST -1 1 " + to_string(req.id);
            case TaskKind::P_PROC:
            case TaskKind::D_PROC:
                fail("cloud task appeared in the edge queue");
        }
        fail("unreachable edge task kind");
    }

In [8]:
display(Markdown("### The central dispatch decision"))
display(
    Code(
        source_between(
            baseline_source, "vector<string> dispatch_ready_work()", "void print_response"
        ),
        language="cpp",
    )
)

### The central dispatch decision

vector<string> dispatch_ready_work() {
        vector<string> assignments;
        assignments.reserve(cloud_count_ + 1);

        if (!edge_busy_) {
            const bool admission_available =
                !pending_requests_.empty() && !free_clouds_.empty();
            const bool edge_task_available = !edge_ready_.empty();

            if (admission_available || edge_task_available) {
                bool choose_admission = false;
                if (!edge_task_available) {
                    choose_admission = true;
                } else if (admission_available) {
                    const Request& pending = request(pending_requests_.front());
                    choose_admission =
                        pending.admission_sequence <= edge_ready_.front().sequence;
                }

                if (choose_admission) {
                    assignments.push_back(dispatch_admission());
                } else {
                    assignments.push_back(dispatch_edge_task());
                }
            }
        }

        for (int cloud = 0; cloud < cloud_count_; ++cloud) {
            if (!cloud_busy_[cloud] && !cloud_ready_[cloud].empty()) {
                assignments.push_back(dispatch_cloud_task(cloud));
            }
        }

        if (assignments.size() > static_cast<size_t>(cloud_count_ + 1)) {
            fail("attempted too many assignments in one response");
        }
        return assignments;
    }

### What makes this a baseline?

The frozen v0 code intentionally does **none** of the following:

- multiple active requests on one cloud;
- load-aware cloud selection;
- grouped decode tasks;
- task-time-table-based batch selection;
- SLO-aware task priority;
- controlled waiting to form a better group;
- layer-chunked prefill; or
- indirect link-aware scheduling.

That is useful experimentally: every later layer has one primary mechanism and a scenario
designed to make that mechanism visible. The layered engine uses compile-time feature gates,
so `OPT_LEVEL=4` contains layers 1 through 4 but none of layers 5 through 7.

## 4. Understand the task-time table

For prefill columns, the lookup size is the request's input length. For decode columns, it
is the decode group size. Missing values (`-1`) are ignored and intermediate sizes are
linearly interpolated by the local judge.

In [9]:
def task_rows_for(scenario_path: Path, scenario: dict[str, Any]) -> list[dict[str, float]]:
    if "task_times" in scenario:
        return scenario["task_times"]
    profile_path = scenario_path.parent / scenario["task_times_file"]
    return json.loads(profile_path.read_text())["task_times"]


task_rows = task_rows_for(selected_path, selected)
display_table(task_rows[:8])

batch_size,prefill_pre,prefill_proc,prefill_post,decode_pre,decode_proc,decode_post
1,0.3,2.0,0.3,0.5,3.0,0.4
4,0.35,3.0,0.32,0.9,7.0,0.7
8,0.4,4.5,0.35,1.3,11.0,1.0
16,0.5,7.0,0.4,2.0,18.0,1.5
32,0.7,12.0,0.5,3.2,30.0,2.4
64,1.0,22.0,0.7,5.5,52.0,4.0
256,2.5,80.0,1.5,18.0,180.0,14.0
4096,20.0,1200.0,10.0,200.0,2000.0,150.0


In [10]:
TASK_COLUMNS = (
    "prefill_pre",
    "prefill_proc",
    "prefill_post",
    "decode_pre",
    "decode_proc",
    "decode_post",
)


def interpolate_duration(rows: list[dict[str, float]], column: str, size: int) -> float:
    points = sorted(
        (int(row["batch_size"]), float(row[column]))
        for row in rows
        if float(row[column]) >= 0
    )
    if not points:
        raise ValueError(f"No usable values for {column}")
    if size <= points[0][0]:
        return points[0][1]
    if size >= points[-1][0]:
        return points[-1][1]
    for (left_size, left_value), (right_size, right_value) in zip(points, points[1:]):
        if size == left_size:
            return left_value
        if left_size < size < right_size:
            fraction = (size - left_size) / (right_size - left_size)
            return left_value + fraction * (right_value - left_value)
    raise AssertionError("Interpolation should have returned")


interpolation_demo = [
    {
        "size": size,
        **{column: round(interpolate_duration(task_rows, column, size), 4) for column in TASK_COLUMNS},
    }
    for size in (1, 2, 4, 6, 8, 16)
]
display_table(interpolation_demo)

size,prefill_pre,prefill_proc,prefill_post,decode_pre,decode_proc,decode_post
1,0.3,2.0,0.3,0.5,3.0,0.4
2,0.3167,2.3333,0.3067,0.6333,4.3333,0.5
4,0.35,3.0,0.32,0.9,7.0,0.7
6,0.375,3.75,0.335,1.1,9.0,0.85
8,0.4,4.5,0.35,1.3,11.0,1.0
16,0.5,7.0,0.4,2.0,18.0,1.5


### A first grouping estimate

The following is a **local service-cost estimate**, not a complete scheduler simulation.
For a decode group of size `b`, it adds:

1. the scheduling cost `S` for each of `D PRE`, `D PROC`, and `D POST`;
2. the interpolated task time for those three stages; and
3. one upload and one download for `b` decode items.

It deliberately ignores queueing, overlap with other resources, and time spent waiting for
requests to become group-compatible. Those effects are why we still need the dynamic judge.

In [11]:
def transfer_time_ms(system: dict[str, Any], item_count: int) -> float:
    size_bytes = item_count * int(system["bytes_per_token"])
    return float(system["latency_in_ms"]) + 8.0 * size_bytes / (
        float(system["bandwidth_gbps"]) * 1_000_000.0
    )


def estimated_decode_cycle_ms(
    system: dict[str, Any], rows: list[dict[str, float]], group_size: int
) -> float:
    compute = sum(
        float(system["S"]) + interpolate_duration(rows, column, group_size)
        for column in ("decode_pre", "decode_proc", "decode_post")
    )
    transfers = 2.0 * transfer_time_ms(system, group_size)
    return compute + transfers


candidate_sizes = [size for size in (1, 2, 4, 8, 16) if size <= len(selected["requests"])]
singleton_cycle = estimated_decode_cycle_ms(selected["system"], task_rows, 1)
group_estimates = []
for size in candidate_sizes:
    group_cycle = estimated_decode_cycle_ms(selected["system"], task_rows, size)
    group_estimates.append(
        {
            "group_size": size,
            "estimated cycle ms": round(group_cycle, 4),
            "ms per request": round(group_cycle / size, 4),
            "idealized throughput gain": f"{size * singleton_cycle / group_cycle:.2f}x",
        }
    )
display_table(group_estimates)

group_size,estimated cycle ms,ms per request,idealized throughput gain
1,28.3001,28.3001,1.00x
2,29.8668,14.9334,1.90x
4,33.0003,8.2501,3.43x
8,37.7007,4.7126,6.01x
16,45.9013,2.8688,9.86x


The largest currently ready group often minimizes service time per request, but that does
**not** prove we should always wait for the largest possible group:

- compatible requests may not be ready yet;
- `D PROC` members must belong to the same cloud;
- waiting increases request age and can violate TDR/TPOT targets;
- a larger group consumes a resource for longer and may block urgent work;
- shared-link queueing can dominate the isolated estimate; and
- the task-time table itself may show weak or negative scaling at larger sizes.

The correct loop is therefore: use the table to form a hypothesis, then use the dynamic
judge to measure the complete policy.

## 5. Run the baseline on one case

The local judge sends startup data and event frames to the actual C++ executable. It accepts
any legal policy decision and generates the resulting `TDN`, `XDN`, and `FIN` events. This is
different from replaying one fixed transcript.

In [12]:
selected_result_path = BUILD_DIR / "notebook-selected-result.json"
run_checked(
    [
        "python3",
        "tools/local_judge.py",
        "--solver",
        str(BASELINE_SOLVER),
        "--scenarios",
        str(selected_path),
        "--json-out",
        str(selected_result_path),
    ]
)
selected_result = json.loads(selected_result_path.read_text())[0]
display_table([selected_result])

PASS batch_friendly_burst         score= 200.433 tp=0.052597 tdr=969.725 tpot=68.663 elapsed=2433.600


scenario,description,legal,score,throughput,tdr,tpot,distance,elapsed,tokens,frames
batch_friendly_burst,A high-overhead request burst where decode grouping should materially improve throughput.,True,200.43253364659435,0.05259697567389861,969.7250000000009,68.6625000000002,0.6162083333333349,2433.6000000000067,128,721


### Reconstruct the score

The score combines normalized throughput with an SLO-compliance component. Higher score and
throughput are better; lower TDR, TPOT, distance, and elapsed time are better.

In [13]:
def clamp01(value: float) -> float:
    return max(0.0, min(1.0, value))


def reconstruct_score(result: dict[str, Any], scoring: dict[str, Any]) -> dict[str, float]:
    excess_tdr = max(0.0, (result["tdr"] - scoring["SLO1"]) / scoring["SLO1"])
    excess_tpot = max(0.0, (result["tpot"] - scoring["SLO2"]) / scoring["SLO2"])
    distance = math.hypot(excess_tdr, excess_tpot)
    throughput_component = clamp01(
        (result["throughput"] - scoring["tp_base"])
        / (scoring["tp_UB"] - scoring["tp_base"])
    )
    distance_base = scoring["dist_base"]
    waiting_component = (
        max(0.0, 1.0 - distance / distance_base)
        if distance_base > 0
        else (1.0 if distance == 0 else 0.0)
    )
    score = 1000.0 * (
        scoring["w_tp"] * throughput_component + scoring["w_c"] * waiting_component
    )
    return {
        "throughput component": throughput_component,
        "SLO distance": distance,
        "SLO component": waiting_component,
        "reconstructed score": score,
    }


score_parts = reconstruct_score(selected_result, selected["scoring"])
display_table([{key: round(value, 6) for key, value in score_parts.items()}])
assert math.isclose(
    score_parts["reconstructed score"], selected_result["score"], rel_tol=0, abs_tol=1e-7
)

throughput component,SLO distance,SLO component,reconstructed score
0.125285,0.616208,0.876758,200.432534


## 6. Run the complete scenario suite

In [14]:
suite_result_path = BUILD_DIR / "notebook-baseline-results.json"
run_checked(
    [
        "python3",
        "tools/local_judge.py",
        "--solver",
        str(BASELINE_SOLVER),
        "--scenarios",
        str(SCENARIO_DIR),
        "--json-out",
        str(suite_result_path),
    ]
)
suite_results = json.loads(suite_result_path.read_text())

display_table(
    [
        {
            "scenario": row["scenario"],
            "score": f"{row['score']:.3f}",
            "throughput": f"{row['throughput']:.6f}",
            "TDR": f"{row['tdr']:.3f}",
            "TPOT": f"{row['tpot']:.3f}",
            "elapsed": f"{row['elapsed']:.3f}",
        }
        for row in suite_results
    ]
)

PASS official_worked_example      score= 500.000 tp=0.022222 tdr=30.000 tpot=0.000 elapsed=45.000
PASS single_sanity                score= 754.111 tp=0.081151 tdr=10.263 tpot=8.902 elapsed=36.968
PASS two_cloud_parallel           score= 776.939 tp=0.127224 tdr=90.035 tpot=10.950 elapsed=227.944
PASS output_length_skew           score= 724.137 tp=0.128046 tdr=96.372 tpot=10.538 elapsed=437.344
PASS batch_friendly_burst         score= 200.433 tp=0.052597 tdr=969.725 tpot=68.663 elapsed=2433.600
PASS latency_sensitive_stream     score= 591.985 tp=0.158490 tdr=106.847 tpot=14.694 elapsed=378.572
PASS link_bottleneck              score= 243.119 tp=0.001185 tdr=14669.700 tpot=227.167 elapsed=27013.300
PASS prefill_preemption           score= 704.428 tp=0.097651 tdr=362.957 tpot=8.907 elapsed=655.396
PASS degenerate_one_layer         score= 766.371 tp=0.084584 tdr=25.970 tpot=7.901 elapsed=70.936
PASS interpolation_missing_values score= 761.047 tp=0.109198 tdr=42.289 tpot=8.909 elapsed=91.577

scenario,score,throughput,TDR,TPOT,elapsed
official_worked_example,500.000,0.022222,30.000,0.000,45.000
single_sanity,754.111,0.081151,10.263,8.902,36.968
two_cloud_parallel,776.939,0.127224,90.035,10.950,227.944
output_length_skew,724.137,0.128046,96.372,10.538,437.344
batch_friendly_burst,200.433,0.052597,969.725,68.663,2433.600
latency_sensitive_stream,591.985,0.158490,106.847,14.694,378.572
link_bottleneck,243.119,0.001185,14669.700,227.167,27013.300
prefill_preemption,704.428,0.097651,362.957,8.907,655.396
degenerate_one_layer,766.371,0.084584,25.970,7.901,70.936
interpolation_missing_values,761.047,0.109198,42.289,8.909,91.577


### Check reproducibility against the saved baseline

In [15]:
saved_baseline = {row["scenario"]: row for row in json.loads(BASELINE_SNAPSHOT.read_text())}
current_baseline = {row["scenario"]: row for row in suite_results}

comparison_rows = []
maximum_score_delta = 0.0
for scenario_name, expected in saved_baseline.items():
    actual = current_baseline[scenario_name]
    score_delta = actual["score"] - expected["score"]
    maximum_score_delta = max(maximum_score_delta, abs(score_delta))
    comparison_rows.append(
        {
            "scenario": scenario_name,
            "legal": actual["legal"],
            "score delta": f"{score_delta:+.9f}",
            "throughput delta": f"{actual['throughput'] - expected['throughput']:+.9f}",
        }
    )

display_table(comparison_rows)
assert all(row["legal"] for row in suite_results)
assert maximum_score_delta < 1e-6
print("Reproducibility check passed: frozen v0 results match baseline-v0.json.")

scenario,legal,score delta,throughput delta
official_worked_example,True,+0.000000000,+0.000000000
single_sanity,True,+0.000000000,+0.000000000
two_cloud_parallel,True,+0.000000000,+0.000000000
output_length_skew,True,+0.000000000,+0.000000000
batch_friendly_burst,True,+0.000000000,+0.000000000
latency_sensitive_stream,True,+0.000000000,+0.000000000
link_bottleneck,True,+0.000000000,+0.000000000
prefill_preemption,True,+0.000000000,+0.000000000
degenerate_one_layer,True,+0.000000000,+0.000000000
interpolation_missing_values,True,+0.000000000,+0.000000000


Reproducibility check passed: frozen v0 results match baseline-v0.json.


## 7. Optimization ladder

The current scheduler implements these changes cumulatively. Each registered version builds
the same layered engine with a different `OPT_LEVEL`, which keeps every comparison attributable
to one newly enabled policy layer.

| Step | Implemented change | Primary scenarios | Expected signal |
|---:|---|---|---|
| 0 | FIFO singleton baseline | all | legal reference point |
| 1 | Allow multiple unfinished requests per cloud | `two_cloud_parallel`, `output_length_skew` | less cloud idle time, lower elapsed time |
| 2 | Assign new requests using current cloud load | `output_length_skew` | less reservation/load imbalance |
| 3 | Group decode-ready work immediately | `batch_friendly_burst` | higher throughput and score |
| 4 | Select group sizes from the task-time table | `nonmonotonic_batch_table`, interpolation | avoid groups whose per-item service rate is worse |
| 5 | Add conservative SLO urgency and bounded waiting | `slo_priority_collision`, `latency_sensitive_stream` | protect aged requests without destroying throughput |
| 6 | Split long `P PROC` work into layer pieces | `single_cloud_prefill_interleave` | let ready decode work interleave between pieces |
| 7 | Add score- and link-aware ordering/group cost | `latency_weighted_slow_link` | improve TDR when latency dominates the score |

The benchmark workbench measures both each version versus v0 and each layer versus the layer
immediately before it. That second comparison is the cleanest local evidence for the effect
of one feature gate.

In [16]:
registry = json.loads(REGISTRY_PATH.read_text())
display_table(
    [
        {
            "layer": version.get("layer", 0),
            "version": version["name"],
            "compile gate": ", ".join(version.get("compile_defines", [])) or "standalone",
            "description": version["description"],
        }
        for version in registry["versions"]
        if version["name"] != "working-tree"
    ]
)

layer,version,compile gate,description
0,v0-baseline,standalone,Frozen FIFO singleton baseline before scoring optimizations.
1,v1-multi-active,OPT_LEVEL=1,Multiple unfinished singleton requests per cloud with round-robin assignment.
2,v2-load-aware,OPT_LEVEL=2,"Adds cloud assignment using busy time, known prefill work, ready decode work, and an active-request proxy."
3,v3-immediate-groups,OPT_LEVEL=3,"Adds immediate grouping for ready D PRE, same-cloud D PROC, and D POST work."
4,v4-table-groups,OPT_LEVEL=4,Selects decode group size by interpolated task-table service rate instead of always taking every ready member.
5,v5-slo-aware,OPT_LEVEL=5,Adds conservative SLO urgency and tightly gated controlled waiting when future events are known.
6,v6-prefill-chunks,OPT_LEVEL=6,Adds adaptive gap-free P PROC layer chunks when longer prefills compete with other cloud work.
7,v7-link-aware,OPT_LEVEL=7,Adds transfer-aware group cost and latency-weighted shortest-prefill ordering while preserving FIFO on throughput-weighted links.


### Where the optimization decisions live

These bounded excerpts are the decision points—not copies of the whole scheduler. Rerunning
the notebook always reads the checked-in C++.

In [17]:
optimization_excerpts = [
    ("Load-aware cloud selection (layer 2)", "int choose_cloud()", "int best_group_size"),
    ("Table-aware group size (layer 4)", "int best_group_size", "bool should_wait_for_group"),
    ("Request urgency (layer 5)", "double request_urgency", "int edge_stage_rank"),
    ("Bounded waiting (layer 5)", "bool should_wait_for_group", "bool should_defer_prefill_admission"),
    ("Adaptive prefill chunks (layer 6)", "int choose_prefill_piece_end", "vector<Candidate> cloud_candidates"),
    ("Latency-weighted prefill ordering (layer 7)", "int take_link_aware_prefill_request", "double cloud_load_score"),
]
for title, start_marker, end_marker in optimization_excerpts:
    display(Markdown(f"#### {title}"))
    display(Code(source_between(layered_source, start_marker, end_marker, max_lines=90), language="cpp"))

#### Load-aware cloud selection (layer 2)

int choose_cloud() {
        if constexpr (kOptimizationLevel == 1) {
            const int cloud = next_round_robin_cloud_;
            next_round_robin_cloud_ = (next_round_robin_cloud_ + 1) % cloud_count_;
            return cloud;
        }

        int best_cloud = 0;
        double best_load = cloud_load_score(0);
        for (int cloud = 1; cloud < cloud_count_; ++cloud) {
            const double load = cloud_load_score(cloud);
            if (load + 1e-12 < best_load) {
                best_load = load;
                best_cloud = cloud;
            }
        }
        return best_cloud;
    }

    double request_urgency(TaskKind kind, const Request& req) const {
        if (kind == TaskKind::P_PRE || kind == TaskKind::P_POST || kind == TaskKind::P_PROC) {
            return (current_time_ - req.arrival_time) / max(1e-9, slo_tdr_);
        }
        return (current_time_ - req.decode_clock_start) / max(1e-9, slo_tpot_);
    }

    int edge_stage_rank(TaskKind kind) const {
        switch (kind) {
            case TaskKind::D_POST:
                return 0;
            case TaskKind::P_POST:
                return 1;
            case TaskKind::D_PRE:
                return 2;
            case TaskKind::P_PRE:
                return 3;
            case TaskKind::P_PROC:
            case TaskKind::D_PROC:
                break;
        }
        return 4;
    }

    vector<Candidate> edge_candidates() {
        vector<Candidate> candidates;
        auto add = [&](TaskKind kind, deque<int>& queue, RequestState expected) {
            if (!queue_available(queue, expected)) {
                return;
            }
            const int request_id = queue.front();
            const Request& req = request(request_id);
            candidates.push_back(
                {kind, request_id, req.ready_sequence, request_urgency(kind, req),
                 edge_stage_rank(kind)}
            );
        };
        add(TaskKind::P_PRE, p_pre_ready_, RequestState::READY_P_PRE);
        add(TaskKind::P_POST, p_post_ready_, RequestState::READY_P_POST);
        add(TaskKind::D_PRE, d_pre_ready_, RequestState::READY_D_PRE);
        add(TaskKind::D_POST, d_post_ready_, RequestState::READY_D_POST);

        if constexpr (kOptimizationLevel < 5) {
            sort(candidates.begin(), candidates.end(), [](const Candidate& left, const Candidate& right) {
                if (left.sequence != right.sequence) {
                    return left.sequence < right.sequence;
                }
                return left.stage_rank < right.stage_rank;
            });
        } else {
            sort(candidates.begin(), candidates.end(), [this](const Candidate& left, const Candidate& right) {
                const bool left_overdue = latency_weight_ > 0.8 && left.urgency >= 1.0;
                const bool right_overdue = latency_weight_ > 0.8 && right.urgency >= 1.0;
                if (left_overdue != right_overdue) {
                    return left_overdue;
                }
                if (left_overdue) {
                    const double left_priority = left.urgency + 0.1 * (3 - left.stage_rank);
                    const double right_priority = right.urgency + 0.1 * (3 - right.stage_rank);
                    if (abs(left_priority - right_priority) > 1e-12) {
                        return left_priority > right_priority;
                    }
                }
                if (left.sequence != right.sequence) {
                    return left.sequence < right.sequence;
                }
                return left.stage_rank < right.stage_rank;
            });
        }
        if constexpr (kOptimizationLevel >= 7) {
            bool link_constrained = bandwidth_gbps_ < 0.1;
// ... bounded notebook preview ...

#### Table-aware group size (layer 4)

int best_group_size(DurationColumn column, int available) const {
        if (available <= 1 || kOptimizationLevel < 3) {
            return 1;
        }
        if constexpr (kOptimizationLevel == 3) {
            return available;
        }

        set<int> candidates = {1, available};
        for (const auto& [size, ignored] : duration_curves_[static_cast<int>(column)]) {
            (void)ignored;
            for (int candidate : {size - 1, size, size + 1}) {
                if (1 <= candidate && candidate <= available) {
                    candidates.insert(candidate);
                }
            }
        }

        int best_size = 1;
        double best_rate = -1;
        for (int size : candidates) {
            double service_time = schedule_cost_ + duration(column, size);
            if constexpr (kOptimizationLevel >= 7) {
                if (column == DurationColumn::DECODE_PRE ||
                    column == DurationColumn::DECODE_PROC) {
                    service_time += transfer_time(
                        static_cast<long long>(size) * bytes_per_token_
                    );
                }
            }
            const double rate = size / service_time;
            if (rate > best_rate + 1e-12 ||
                (abs(rate - best_rate) <= 1e-12 && size < best_size)) {
                best_rate = rate;
                best_size = size;
            }
        }
        return best_size;
    }

#### Request urgency (layer 5)

...

bool should_wait_for_group(
        TaskKind kind,
        DurationColumn column,
        int cloud,
        int available,
        double oldest_ready_time,
        bool allow_wait
    ) const {
        if constexpr (kOptimizationLevel < 5) {
            return false;
        }
        if (!allow_wait || kind == TaskKind::D_POST || available <= 0) {
            return false;
        }
        // Waiting has no self-wake timer and easily overshoots on sparse streams. Restrict it
        // to tests that are almost purely throughput-weighted; the urgency guard below still
        // prevents waiting once a member consumes half of its TPOT budget.
        if (throughput_weight_ < 0.95) {
            return false;
        }
        int possible = total_active_decode_requests_;
        if (kind == TaskKind::D_PROC) {
            possible = active_decode_requests_[cloud];
        }
        possible = max(possible, available);
        const int target = best_group_size(column, possible);
        if (available >= target) {
            return false;
        }
        const Request& oldest = request(
            kind == TaskKind::D_PROC ? d_proc_ready_[cloud].front() : d_pre_ready_.front()
        );
        if (request_urgency(kind, oldest) >= 0.5) {
            return false;
        }
        const double wait_budget = slo_tpot_ * (0.02 + 0.18 * throughput_weight_);
        const double waited = current_time_ - oldest_ready_time;
        return waited + 1e-12 < wait_budget && has_known_future_event();
    }

#### Adaptive prefill chunks (layer 6)

int choose_prefill_piece_end(const Request& req, int cloud) const {
        if constexpr (kOptimizationLevel < 6) {
            return layer_count_;
        }
        const int remaining_layers = layer_count_ - req.next_prefill_layer;
        if (remaining_layers <= 1 || layer_count_ <= 8) {
            return layer_count_;
        }
        const double full_duration = duration(DurationColumn::PREFILL_PROC, req.input_length);
        const bool competing = !d_proc_ready_[cloud].empty() ||
                               active_decode_requests_[cloud] > 0 ||
                               p_proc_ready_[cloud].size() > 1;
        if (!competing) {
            return layer_count_;
        }
        const double token_multiple = layer_count_ <= 8 ? 2.0 : 0.5;
        const double target_duration = max(
            4.0 * schedule_cost_,
            min(0.25 * slo_tdr_, token_multiple * slo_tpot_)
        );
        int piece_layers = static_cast<int>(ceil(
            target_duration * layer_count_ / max(1e-12, full_duration)
        ));
        piece_layers = max(1, min(piece_layers, remaining_layers));
        return req.next_prefill_layer + piece_layers;
    }

#### Latency-weighted prefill ordering (layer 7)

int take_link_aware_prefill_request() {
        clean_front(p_pre_ready_, RequestState::READY_P_PRE);
        if (p_pre_ready_.empty()) {
            fail("link-aware admission read an empty queue");
        }
        if constexpr (kOptimizationLevel < 7) {
            const int request_id = p_pre_ready_.front();
            p_pre_ready_.pop_front();
            return request_id;
        }
        if (latency_weight_ <= throughput_weight_) {
            const int request_id = p_pre_ready_.front();
            p_pre_ready_.pop_front();
            return request_id;
        }

        const int window = min<int>(64, p_pre_ready_.size());
        int best_index = 0;
        double best_score = numeric_limits<double>::infinity();
        for (int index = 0; index < window; ++index) {
            const Request& req = request(p_pre_ready_[index]);
            const double age_ratio = (current_time_ - req.arrival_time) / max(1e-9, slo_tdr_);
            const double transfer = transfer_time(
                static_cast<long long>(req.input_length) * bytes_per_token_
            );
            const double score = transfer - age_ratio * slo_tdr_ * 0.5;
            if (score < best_score) {
                best_score = score;
                best_index = index;
            }
        }
        const int request_id = p_pre_ready_[best_index];
        p_pre_ready_.erase(p_pre_ready_.begin() + best_index);
        return request_id;
    }

### Optimization experiment worksheet

Copy this template into a new Markdown cell for each change:

```text
Optimization:
Hypothesis:
Code path changed:
Correctness invariant at risk:
Primary scenario:
Expected metric movement:
Observed result:
Interpretation:
Keep, revise, or revert:
```

## 8. Compare the current layer-7 scheduler with v0

This compact comparison answers “did the accumulated policy help?” The companion benchmark
notebook performs the more diagnostic v0 → v1 → ... → v7 comparison.

In [18]:
def safe_label(label: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "-", label).strip("-")


def run_policy_suite(label: str, executable: Path) -> list[dict[str, Any]]:
    executable = executable.resolve()
    if not executable.is_file():
        raise FileNotFoundError(executable)
    result_path = BUILD_DIR / f"notebook-{safe_label(label)}-results.json"
    run_checked(
        [
            "python3",
            "tools/local_judge.py",
            "--solver",
            str(executable),
            "--scenarios",
            str(SCENARIO_DIR),
            "--json-out",
            str(result_path),
        ]
    )
    return json.loads(result_path.read_text())


policy_results: dict[str, list[dict[str, Any]]] = {
    "baseline-v0": suite_results,
    "current-v7": run_policy_suite("current-v7", WORKING_SOLVER),
}

print("Loaded policies:", ", ".join(policy_results))

PASS official_worked_example      score= 500.000 tp=0.022222 tdr=30.000 tpot=0.000 elapsed=45.000
PASS single_sanity                score= 754.111 tp=0.081151 tdr=10.263 tpot=8.902 elapsed=36.968
PASS two_cloud_parallel           score=1000.000 tp=0.242143 tdr=28.048 tpot=13.604 elapsed=119.764
PASS output_length_skew           score= 708.206 tp=0.122735 tdr=23.249 tpot=11.294 elapsed=456.267
PASS batch_friendly_burst         score= 676.030 tp=0.227611 tdr=208.020 tpot=36.179 elapsed=562.363
PASS latency_sensitive_stream     score= 996.673 tp=0.279193 tdr=20.848 tpot=18.374 elapsed=214.905
PASS link_bottleneck              score= 241.109 tp=0.001396 tdr=12857.167 tpot=371.167 elapsed=22921.400
PASS prefill_preemption           score= 730.802 tp=0.097705 tdr=159.289 tpot=10.603 elapsed=655.035
PASS degenerate_one_layer         score=1000.000 tp=0.159849 tdr=9.159 tpot=7.967 elapsed=37.535
PASS interpolation_missing_values score= 879.952 tp=0.154382 tdr=26.086 tpot=10.144 elapsed=64.774


In [19]:
def comparison_against_baseline(
    baseline_rows: list[dict[str, Any]], candidate_rows: list[dict[str, Any]]
) -> list[dict[str, Any]]:
    baseline_map = {row["scenario"]: row for row in baseline_rows}
    candidate_map = {row["scenario"]: row for row in candidate_rows}
    rows = []
    for scenario_name, old in baseline_map.items():
        new = candidate_map[scenario_name]
        if not new.get("legal", False):
            rows.append({"scenario": scenario_name, "status": "ILLEGAL"})
            continue
        rows.append(
            {
                "scenario": scenario_name,
                "status": "legal",
                "score delta": f"{new['score'] - old['score']:+.3f}",
                "throughput %": f"{100 * (new['throughput'] / old['throughput'] - 1):+.1f}%",
                "TDR %": f"{100 * (new['tdr'] / old['tdr'] - 1):+.1f}%" if old["tdr"] else "n/a",
                "TPOT %": f"{100 * (new['tpot'] / old['tpot'] - 1):+.1f}%" if old["tpot"] else "n/a",
                "elapsed %": f"{100 * (new['elapsed'] / old['elapsed'] - 1):+.1f}%",
            }
        )
    return rows


for policy_name, results in policy_results.items():
    if policy_name == "baseline-v0":
        continue
    display(Markdown(f"### {policy_name} vs baseline-v0"))
    display_table(comparison_against_baseline(suite_results, results))

### current-v7 vs baseline-v0

scenario,status,score delta,throughput %,TDR %,TPOT %,elapsed %
official_worked_example,legal,+0.000,+0.0%,+0.0%,n/a,+0.0%
single_sanity,legal,+0.000,+0.0%,+0.0%,+0.0%,+0.0%
two_cloud_parallel,legal,+223.061,+90.3%,-68.8%,+24.2%,-47.5%
output_length_skew,legal,-15.931,-4.1%,-75.9%,+7.2%,+4.3%
batch_friendly_burst,legal,+475.597,+332.7%,-78.5%,-47.3%,-76.9%
latency_sensitive_stream,legal,+404.688,+76.2%,-80.5%,+25.0%,-43.2%
link_bottleneck,legal,-2.010,+17.9%,-12.4%,+63.4%,-15.1%
prefill_preemption,legal,+26.374,+0.1%,-56.1%,+19.0%,-0.1%
degenerate_one_layer,legal,+233.629,+89.0%,-64.7%,+0.8%,-47.1%
interpolation_missing_values,legal,+118.905,+41.4%,-38.3%,+13.9%,-29.3%


## Checks

These assertions verify the lab itself:

- every registered scenario was discovered;
- every baseline and current-policy interaction was legal;
- the official calibration case reproduces 45 ms elapsed and TDR 30 ms;
- the score formula reconstruction matched the judge; and
- frozen v0 results matched the saved `baseline-v0` snapshot.

In [20]:
assert len(scenario_paths) >= 14
assert all(row["legal"] for row in suite_results)
assert all(row["legal"] for row in policy_results["current-v7"])
assert set(current_baseline) == set(saved_baseline)

official = current_baseline["official_worked_example"]
assert math.isclose(official["elapsed"], 45.0, abs_tol=1e-9)
assert math.isclose(official["tdr"], 30.0, abs_tol=1e-9)
assert math.isclose(official["tpot"], 0.0, abs_tol=1e-9)

run_checked(["make", "transcript-test"])
print("All notebook checks passed.")

python3 tests/run_transcript_tests.py build/v0-baseline
PASS official_example
PASS two_cloud_fifo
All notebook checks passed.


## Next steps

Open `scheduler_benchmark_workbench.ipynb` next. Its incremental table tells us which exact
layer moved which exact scenario. Use that evidence to tune one feature gate at a time:

1. inspect the target scenario and its score components;
2. review any scenario-level regression, even when the suite mean rose;
3. change one threshold or policy rule;
4. rerun both legality validation and the full version workbench; and
5. keep the change only when its tradeoff matches the supplied scoring weights.

The local scenarios are deterministic mechanism tests, not a model of the official hidden
workload distribution. A local win is evidence that a mechanism works under those inputs;
it is not a guarantee of leaderboard improvement.